# Lack of fit and pure error

When an input is repeated, the responses at that input differ by noise rather than by the polynomial. Those within-input differences estimate $\sigma^2$ without assuming a degree. The rest of the residual sum is lack of fit: the part a richer polynomial might still explain.


## Three inputs, two replicates

$$
\begin{array}{c|cc|cc|cc}
x & 0 & 0 & 1 & 1 & 2 & 2 \\
\hline
y & -1 & 1 & -1 & 1 & 11 & 13
\end{array}
$$

The three pair means are $0$, $0$, and $12$. Inside each pair the responses sit one unit on either side of the mean, so the pure-error sum of squares is

$$
\mathrm{SS}_{\mathrm{pe}} = 6\cdot 1^2 = 6,
$$

on $3$ degrees of freedom, one for each pair. A mean at a repeated input costs one parameter and leaves one contrast. This estimate does not use a line or a parabola. Any polynomial, evaluated at a fixed $x$, predicts the same number for both replicates, so it cannot absorb $\mathrm{SS}_{\mathrm{pe}}$.


## The line

The sums are $n = 6$, $\sum x = 6$, $\sum y = 24$, $\sum xy = 48$, and $\sum x^2 = 10$.

$$
\begin{aligned}
b_1 &= \dfrac{6\cdot 48 - 6\cdot 24}{6\cdot 10 - 6^2} = \dfrac{144}{24} = 6, \\
b_0 &= \dfrac{24 - 6\cdot 6}{6} = -2.
\end{aligned}
$$

The line $-2 + 6x$ predicts $-2, -2, 4, 4, 10, 10$. The residuals are $1, -3, -5, -3, 1, 3$, and

$$
\mathrm{RSS}_1 = 1 + 9 + 25 + 9 + 1 + 9 = 54.
$$

Lack of fit is what remains after pure error is removed:

$$
\mathrm{SS}_{\mathrm{lof}} = 54 - 6 = 48.
$$

There are $3$ distinct inputs and $2$ mean parameters in the line, so lack of fit has $3 - 2 = 1$ degree of freedom. The test statistic is

$$
F = \dfrac{48/1}{6/3} = 24.
$$

The $0.95$ quantile of $F_{1, 3}$ is about $10.13$. The line's lack of fit is rejected. The pair means $0, 0, 12$ are not in arithmetic progression, and the pure-error variance $2$ is too small to explain a residual sum of $54$.


In [1]:
# The line -2 + 6x has RSS 54. Lack of fit over pure error is F = 24.
import scipy.stats as stats
x = [0, 0, 1, 1, 2, 2]
y = [-1, 1, -1, 1, 11, 13]
fitted = [-2 + 6 * xi for xi in x]
rss = sum((yi - yh) ** 2 for yi, yh in zip(y, fitted))
pure = 6
lack = rss - pure
f_stat = (lack / 1) / (pure / 3)
critical = stats.f.ppf(0.95, 1, 3)
print("RSS", rss, "lack of fit", lack, "F", f_stat, "critical", critical)
assert fitted == [-2, -2, 4, 4, 10, 10]
assert rss == 54 and lack == 48 and f_stat == 24
assert f_stat > critical


RSS 54 lack of fit 48 F 24.0 critical 10.127964486013925


## The quadratic hits the means

The unique polynomial of degree at most $2$ through the means $(0, 0)$, $(1, 0)$, and $(2, 12)$ is

$$
6x^2 - 6x.
$$

Check: at $x = 0$ the value is $0$, at $x = 1$ it is $0$, and at $x = 2$ it is $24 - 12 = 12$. The six residuals are then exactly the within-pair deviations $-1, 1, -1, 1, -1, 1$. Thus

$$
\mathrm{RSS}_2 = 6 = \mathrm{SS}_{\mathrm{pe}}, \qquad \mathrm{SS}_{\mathrm{lof}} = 0.
$$

Three distinct inputs and a quadratic have no lack-of-fit degrees of freedom left: $3 - 3 = 0$. The quadratic is the saturated model for these means. Its residual sum is pure error, and there is no further $F$ test against a richer polynomial in $x$. Any such polynomial would still predict one number per input and could not reduce $\mathrm{SS}_{\mathrm{pe}}$.


In [2]:
# The quadratic 6x^2 - 6x fits the three means and leaves only pure error.
x = [0, 0, 1, 1, 2, 2]
y = [-1, 1, -1, 1, 11, 13]
fitted = [6 * xi ** 2 - 6 * xi for xi in x]
residuals = [yi - yh for yi, yh in zip(y, fitted)]
print("fitted", fitted)
print("residuals", residuals, "RSS", sum(r ** 2 for r in residuals))
assert fitted == [0, 0, 0, 0, 12, 12]
assert residuals == [-1, 1, -1, 1, -1, 1]
assert sum(r ** 2 for r in residuals) == 6


fitted [0, 0, 0, 0, 12, 12]
residuals [-1, 1, -1, 1, -1, 1] RSS 6


## Pitfall

Pure error requires genuine replicates: two responses at the same $x$, not two responses at neighboring values that were rounded to the same recorded $x$. If the recorded ties are really distinct inputs, the within-pair sum contains curvature and is not an estimate of $\sigma^2$ alone. The split $\mathrm{RSS} = \mathrm{SS}_{\mathrm{lof}} + \mathrm{SS}_{\mathrm{pe}}$ is an algebraic identity whenever the fitted polynomial is constant on each replicated input. The interpretation of $\mathrm{SS}_{\mathrm{pe}}$ as noise is a statement about how the replicates were obtained.


In [3]:
# The pure-error sum is the sum of squared deviations from the pair means.
pairs = [(-1, 1), (-1, 1), (11, 13)]
pure = 0
for left, right in pairs:
    mean = (left + right) / 2
    pure += (left - mean) ** 2 + (right - mean) ** 2
print("pure error", pure)
assert pure == 6


pure error 6.0


## Summary

- Three replicated pairs give $\mathrm{SS}_{\mathrm{pe}} = 6$ on $3$ degrees of freedom.
- The line $-2 + 6x$ has $\mathrm{RSS} = 54$, so lack of fit is $48$ and $F = 24$, above the $F_{1,3}$ critical value.
- The quadratic $6x^2 - 6x$ fits the three means. Its residual sum equals pure error, and no lack-of-fit degrees of freedom remain.
